In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import subprocess
import tempfile
from pathlib import Path

NUM_TASKS = 500
BASE_PATH = "../logs/swebench/llamacpp_qwen3_coder_30b"
MODEL = "Qwen3-Coder-30B-A3B"

In [ ]:
df = pd.read_parquet("hf://datasets/SWE-bench/SWE-bench_Verified/data/test-00000-of-00001.parquet")
git_diffs = []
for i in range(NUM_TASKS):
    with open(f"{BASE_PATH}/{i}/run_0/analysis/summary.json") as fp:
        summary = json.load(fp)
        output = summary["agent_output"]
        if output is None:
            print(f"{i}: None")
        elif "<output>" not in output or "</output>" not in output:
            print(f"{i}: Missing output tag: {output}")
        if output is None or "<output>" not in output or "</output>" not in output:
            git_diffs.append(None)
        else:
            diff = output[(output.index("<output>") + 10):output.index("</output>")]
            pattern = re.compile(r"\\[^n]")
            diff = bytes(diff, "utf-8").decode("unicode_escape")
            stop_sign = "COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT"
            if diff.startswith(stop_sign):
                diff = diff[len(stop_sign):]

            diff_git_idx = diff.find("diff --git")
            if diff_git_idx == -1:
                git_diffs.append(None)
                print(f"{i}: Empty")
                continue
            
            diff = diff[diff_git_idx:]
            lines = diff.split("\n")
            lines = [l for l in lines if not l.startswith("index ")]
            diff = "\n".join(lines)
            git_diffs.append(diff)

In [ ]:
instance_ids = []
predictions = []

same = 0
for i in range(NUM_TASKS):
    diff = git_diffs[i]
    if git_diffs[i] is None:
        continue
    if diff != df["patch"].iloc[i]:
        id = df["instance_id"].iloc[i]
        instance_ids.append(id)
        predictions.append({"instance_id": id, "model_name_or_path": MODEL, "model_patch": diff})
    else:
        same += 1

print(f"{same} same solutions.")
print(f"{len(instance_ids)} different solutions.")

In [ ]:
with tempfile.NamedTemporaryFile(mode="w+", suffix=".jsonl", delete=False) as tmp:
    tmp.write("\n".join(json.dumps(p) for p in predictions))
    tmp_path = tmp.name

# Set DOCKER_HOST to Colima's socket path (update this path accordingly)
env = os.environ.copy()
env['DOCKER_HOST'] = f"unix://{str(Path.home())}/.colima/default/docker.sock"

cmd = [
    "python", "-m", "swebench.harness.run_evaluation",
    "--dataset_name", "princeton-nlp/SWE-bench_Verified",
    "--run_id", "mini_swe_agent",
    "--max_workers", "4",
    "--clean", "True",
    "--predictions_path", tmp_path,
    "--instance_ids"
] + list(instance_ids)

res = subprocess.run(cmd, env=env, text=True)
print(res)

os.remove(tmp_path)

In [ ]:
is_correct = [False] * NUM_TASKS
for i in range(NUM_TASKS):
    diff = git_diffs[i]
    if git_diffs[i] is None:
        continue
    if diff == df["patch"].iloc[i]:
        is_correct[i] = True
        continue
    id = df['instance_id'].iloc[i]
    try:
        with open(f"logs/run_evaluation/mini_swe_agent/{MODEL}/{id}/report.json", "r") as fp:
            res = json.load(fp)
            resolved = res[id]["resolved"]
            if resolved:
                is_correct[i] = True
    except:
        print(f"Instance {id} does not have report.json")
        continue

print(f"{sum(is_correct)}/{len(is_correct)}")

In [ ]:
# Load profiling and logprob data
regex = r"logprob=(-?\d+\.\d+(?:[eE][-+]?\d+)?)"
token_regex = r"ChatCompletionTokenLogprob\(token=(['\"])(.*?)\1, bytes="
all_logprobs = []
all_tokens = []
for i in range(NUM_TASKS):
    with open(f"{BASE_PATH}/{i}/run_0/raw/trace.json", "r") as f:
        trace = json.load(f)
        trace = [t for t in trace if t["name"] == "ActionStep" and "logprob=" in t["attributes"]["output.value"]]
        trace = sorted(trace, key=lambda t: t["start_time"])
        logprobs = []
        tokens = []
        for t in trace:
            v = json.loads(t["attributes"]["output.value"])
            logprobs.append([
                float(re.search(regex, l).group(1)) for l in v["model_output_message"]["raw"]["logprobs"]
            ])
            tokens.append([
                re.search(token_regex, l).group(2) for l in v["model_output_message"]["raw"]["logprobs"]
            ])
        all_logprobs.append(logprobs)
        all_tokens.append(tokens)


In [ ]:
plt.hist([len(lp) for i, lp in enumerate(all_logprobs) if not is_correct[i]], color="red", alpha=0.5, label="Fail")
plt.hist([len(lp) for i, lp in enumerate(all_logprobs) if is_correct[i]], color="green", alpha=0.8, label="Success")
plt.xlabel("Number of steps")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
def plot_min_logprobs(
    start_step,
    end_step,
    num_logprob=10,
    exp=False,
    trace_type=None,
):
    assert start_step <= end_step, "Start step must not be greater than end step"
    plt.figure(figsize=(16, 5))
    cnt = 0

    for i, logprobs in enumerate(all_logprobs):
        if len(logprobs) <= end_step:
            continue
        if trace_type is not None:
            if is_correct[i] != trace_type:
                continue
        cnt += 1
        color = "green" if is_correct[i] else "red"
        style = "-"
        alpha = 0.8 if is_correct[i] else 0.2

        for step, step_logprobs in enumerate(logprobs[start_step-1:end_step]):
            if exp:
                step_logprobs = np.exp(step_logprobs)

            line = np.sort(step_logprobs)[:num_logprob]

            start = step * num_logprob + 1
            plt.plot(range(start, start + len(line)), line, color=color, alpha=alpha, linestyle=style)

    ax = plt.gca()
    ax.set_xticks([])
    ax.set_xticklabels([])

    plt.xlabel("Step")
    plt.ylabel("Logprobs" if not exp else "Probabilities")
    plt.title(f"Top {num_logprob} smallest token {'logprobs' if not exp else 'probabilities'} vs Step (n = {cnt})")

    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_min_logprobs(1, 5, num_logprob=10)

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

import difflib

def lcs(a: str, b: str) -> str:
    matcher = difflib.SequenceMatcher(None, a, b)
    match = matcher.find_longest_match(0, len(a), 0, len(b))
    if match.size == 0:
        return ""
    return a[match.a: match.a + match.size]

def normalize(arr):
    return (arr - np.mean(arr)) / np.std(arr)

# ------------------ 1) Prepare training data ------------------ #
def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        # num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        # feature_vector.extend(num_thought_tokens)

        # Length of longest common substring between current step and previous step
        # for j in range(clf_step - 1, clf_step):
        #     cur_gen = "".join(tokens[j])
        #     prev_gen = "".join(tokens[j - 1])
        #     feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    X = np.array(X_features)
    y = np.array(y)

    # scaler = StandardScaler()
    # X = scaler.fit_transform(X)
    # poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
    # X = poly.fit_transform(X)

    return X, y, valid

def train_classifier(logprobs_data, tokens_data, labels, clf_step, n_folds=5, num_logprobs=10):
    X, y, valid = extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=num_logprobs)
    print(f"X dimension: {X.shape}")
    print(f"y positive: {y.mean().round(2)} ({y.sum()}/{X.shape[0]})")

    # ------------------ 2) Hyperparameter Search (Nested CV) ------------------ #
    estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    )

    param_grid = {
        "n_estimators": [10, 20, 50, 100, 200, 300],
        "learning_rate": [0.001, 0.005, 0.01, 0.05, 0.1, 0.2],
        "max_depth": [2, 3, 4, 5, 6],
        "min_child_weight": [1, 3, 5, 7, 10],
        # "subsample": [0.8, 0.9, 1.0],
        # "gamma": [0, 0.1, 0.2, 0.3, 0.5],
        # "reg_alpha": [0, 0.01, 0.1, 1, 5, 10],
        # "reg_lambda": [0.1, 0.5, 1, 5, 10]
    }

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42),
        n_jobs=-1,
    )

    search.fit(X, y)
    best_model = search.best_estimator_
    print("Best hyperparameters:", search.best_params_)

    # ------------------ 3) Out-of-Fold (OOF) Predictions ------------------ #
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_proba = np.zeros_like(y, dtype=float)

    for tr_idx, va_idx in cv.split(X, y):
        model = clone(best_model)
        model.fit(X[tr_idx], y[tr_idx])
        proba = model.predict_proba(X[va_idx])[:, 1]
        oof_proba[va_idx] = proba

    # ------------------ 4) Find Best Threshold Using OOF Predictions ------------------ #
    prec, rec, thr = precision_recall_curve(y, oof_proba)
    prec = prec[:-1]
    rec = rec[:-1]
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    best_idx = np.nanargmax(f1)
    best_threshold = thr[best_idx] if best_idx < len(thr) else 0.5
    

    # alpha = 0.0  # higher alpha -> prioritize recall more
    # # Compute custom score
    # score = alpha * rec + (1 - alpha) * prec
    # # Find best threshold
    # best_idx = np.argmax(score)
    # best_threshold = thr[best_idx] if best_idx < len(thr) else 1.0

    print("Best threshold:", round(float(best_threshold), 6))
    oof_preds = (oof_proba >= best_threshold).astype(int)

    # ------------------ 5) Evaluate Performance via CV ------------------ #
    print(f"CV num predicted positive: {oof_preds.sum()}")
    print("CV ROC AUC:", roc_auc_score(y, oof_proba))
    print("CV F1:", f1_score(y, oof_preds))
    print("CV Accuracy:", accuracy_score(y, oof_preds))
    print("CV Precision:", precision_score(y, oof_preds))
    print("CV Recall:", recall_score(y, oof_preds))

    return oof_proba, valid

    # # ------------------ 6) Final Model Training ------------------ #
    # final_model = clone(best_model).fit(X, y)
    # print("Final model trained on full dataset.")
    # return final_model

In [ ]:
labels = [not c for c in is_correct]
for num_logprobs in [1, 2, 3]:
    for num_steps in [5, 6]:
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        oof_prob, valid = train_classifier(all_logprobs, all_tokens, labels, clf_step=num_steps, num_logprobs=num_logprobs)
        all_oof_prob = [None] * NUM_TASKS
        for j, i in enumerate(valid):
            all_oof_prob[i] = oof_prob[j]